In [ ]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths to all ablation result files
# ==========================================================
FILES = {
    "without weather data": "with_without_weather_data/RAG_WEATHER_ABLATION_STATS.csv",
    "without textual representation": "with_without_textual_representation/RAG_RETRIEVAL_ABLATION_STATS.csv",
    "without retrieval": "with_without_retrieval/RAG_WITHOUT_RETRIEVAL_ABLATION_STATS.csv",
    "without feedback loop": "with_without_feedback_loop/RAG_FEEDBACK_ABLATION_STATS.csv",
    "without barometric altitude": "with_without_barometric_altitude/RAG_BAROMETRIC_ABLATION_STATS.csv",
}

OUTPUT_PATH = "ABLATION_STUDY_FULL_TABLE.csv"

# ==========================================================
# Load and normalize all ablation files
# ==========================================================
all_dfs = []

for method_name, file_path in FILES.items():
    df = pd.read_csv(file_path)

    # Add column identifying which ablation it is
    df["method"] = method_name

    all_dfs.append(df)

# Concatenate everything
final_df = pd.concat(all_dfs, ignore_index=True)

# ==========================================================
# Reorder columns to match your table layout
# ==========================================================
desired_order = [
    "phase",
    "method",
    "mean",
    "median",
    "std",
    "Q1",
    "Q2",
    "min",
    "max",
    "p_value"
]

# Keep only columns that exist
final_df = final_df[[col for col in desired_order if col in final_df.columns]]

# ==========================================================
# Sort properly (Take off → Cruising → Landing)
# ==========================================================
phase_order = {"take_off": 0, "cruising": 1, "landing": 2}
final_df["phase_order"] = final_df["phase"].map(phase_order)

final_df = final_df.sort_values(["phase_order", "method"]).drop(columns="phase_order")

# ==========================================================
# Round numeric columns to 2 decimals
# ==========================================================
numeric_cols = final_df.select_dtypes(include=["float64", "int64"]).columns
final_df[numeric_cols] = final_df[numeric_cols].round(2)

# ==========================================================
# Save final merged CSV
# ==========================================================
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Final ablation table saved to: {OUTPUT_PATH}")


In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

# ==========================================================
# File paths
# ==========================================================
FILES = {
    "without_barometric_altitude": "with_without_barometric_altitude/HAVERSINE_ERROR_RAG_WITHOUT_BAROMETRIC_ALTITUDE.json",
    "without_feedback_loop": "with_without_feedback_loop/HAVERSINE_ERROR_RAG_WITHOUT_FEEDBACK_LOOP.json",
    "without_retrieval": "with_without_retrieval/HAVERSINE_ERROR_RAG_WITHOUT_RETRIEVAL.json",
    "without_textual_representation": "with_without_textual_representation/HAVERSINE_ERROR_RAG_WITHOUT_TEXTUAL_REPRESENTATION.json",
    "without_weather_data": "with_without_weather_data/MEAN_HAVERSINE_RAG_WITHOUT_WEATHER.json",
}

RAG_FILE = "../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_RAG.json"

PHASES = ["take_off", "cruising", "landing"]


# ==========================================================
# Helper functions
# ==========================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def extract_errors(data, phase):
    """
    Extract list of errors from JSON.
    Handles both formats:
    - [[error, flight_id], ...]
    - [error1, error2, ...]
    """
    values = data["RAG"][phase]

    # If format is [[error, id], ...]
    if isinstance(values[0], list):
        return [v[0] for v in values]

    # If format is already list of numbers
    return values


def ensure_25_values(lst):
    """
    If length is 24, append mean as 25th value.
    """
    if len(lst) == 24:
        lst = lst.copy()
        lst.append(np.mean(lst))
    return lst


# ==========================================================
# Load baseline RAG
# ==========================================================
rag_data = load_json(RAG_FILE)

results = {}

# ==========================================================
# Loop through all ablation experiments
# ==========================================================
for exp_name, path in FILES.items():

    exp_data = load_json(path)
    results[exp_name] = {}

    for phase in PHASES:

        rag_errors = extract_errors(rag_data, phase)
        exp_errors = extract_errors(exp_data, phase)

        rag_errors = ensure_25_values(rag_errors)
        exp_errors = ensure_25_values(exp_errors)

        # Convert to numpy arrays
        rag_errors = np.array(rag_errors)
        exp_errors = np.array(exp_errors)

        # Check equal length
        if len(rag_errors) != len(exp_errors):
            print(f"Skipping {exp_name} - {phase} (size mismatch)")
            results[exp_name][phase] = None
            continue

        # Wilcoxon signed-rank test (one-sided: RAG < ablation)
        stat, p_value = wilcoxon(rag_errors, exp_errors, alternative="less")

        results[exp_name][phase] = round(p_value, 10)

# ==========================================================
# Print results nicely
# ==========================================================
print("\n===== P-VALUES (H1: RAG has lower error) =====\n")

for exp, phases in results.items():
    print(f"{exp}:")
    for phase, p in phases.items():
        print(f"   {phase}: {p}")
    print()
